# Ground Truth Analysis

This notebook evaluates the considered algorithms on synthetic graphs with a
known ground-truth community structure.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

## Configuration

In [2]:
SIZE_ORDER = ["small", "large"]
REGIME_ORDER = ["sparse", "dense"]
NOISE_ORDER = [0.05, 0.20]

COMMUNITY_SIZE_ORDER = [
    "tiny",
    "small",
    "large",
]

ALGORITHM_ORDER = [
    "leiden",
    "leiden_mdgp",
    "kapoce",
    "mdgp_plateau",
]

DATASET_ORDER = [
    ("small", "sparse"),
    ("small", "dense"),
    ("large", "sparse"),
    ("large", "dense"),
]

COMMUNITY_ORDER_BY_SIZE = {
    "small": ["small", "large"],
    "large": ["tiny", "small", "large"],
}

GROUP_COLUMNS = [
    "size_class",
    "regime",
    "noise",
    "community_size_class",
]

DATA_ROOT = Path("../../data/ground_truth")

RESULTS_DIR = Path("../../results/experiment4")
RAW_RESULTS_FILE = RESULTS_DIR / "raw_results.csv"

## Load data

In [3]:
raw = pd.read_csv(RAW_RESULTS_FILE)

raw["noise"] = raw["noise"].astype(float)

raw["size_class"] = pd.Categorical(
    raw["size_class"],
    categories=SIZE_ORDER,
    ordered=True,
)

raw["regime"] = pd.Categorical(
    raw["regime"],
    categories=REGIME_ORDER,
    ordered=True,
)

raw["community_size_class"] = pd.Categorical(
    raw["community_size_class"],
    categories=COMMUNITY_SIZE_ORDER,
    ordered=True,
)

raw["algorithm"] = pd.Categorical(
    raw["algorithm"],
    categories=ALGORITHM_ORDER,
    ordered=True,
)

## Generated instance characteristics

In [4]:
def minimum_ground_truth_cluster_size(partition_value: str) -> int:
    partition = json.loads(partition_value)
    return min(len(cluster) for cluster in partition)

In [5]:
instance_columns = [
    "size_class",
    "regime",
    "noise",
    "community_size_class",
    "instance",
    "n",
    "m",
    "edge_density",
    "ground_truth_density",
    "ground_truth_num_clusters",
    "ground_truth_max_cluster_size",
    "ground_truth_avg_cluster_size",
    "ground_truth_partition",
]

instances = raw[instance_columns].drop_duplicates(subset=["instance"]).copy()

instances["average_degree"] = 2 * instances["m"] / instances["n"]

instances["ground_truth_min_cluster_size"] = instances["ground_truth_partition"].apply(minimum_ground_truth_cluster_size)

In [6]:
def target_average_degree(n: int, regime: str) -> float:
    if regime == "sparse":
        return 8.0

    if regime == "dense":
        return max(16.0, 0.04 * n)


instances["target_average_degree"] = instances.apply(lambda row: target_average_degree(n=row["n"], regime=str(row["regime"])), axis=1)

degree_summary = (
    instances
    .groupby(["size_class", "regime"], observed=True, as_index=False)
    .agg(
        mean_target_degree=("target_average_degree", "mean"),
        mean_realized_degree=("average_degree", "mean"),
    )
)

degree_summary

,size_class,regime,mean_target_degree,mean_realized_degree
0,small,sparse,8.00000,9.380563
1,small,dense,16.00000,18.291761
2,large,sparse,8.00000,9.800944
3,large,dense,40.52816,47.149936


In [7]:
community_size_summary = (
    instances
    .groupby(GROUP_COLUMNS, observed=True, as_index=False)
    .agg(
        mean_num_clusters=("ground_truth_num_clusters", "mean"),
        mean_cluster_size=("ground_truth_avg_cluster_size", "mean"),
        mean_min_cluster_size=("ground_truth_min_cluster_size", "mean"),
        mean_max_cluster_size=("ground_truth_max_cluster_size", "mean"),
    )
)

community_size_summary

,size_class,regime,noise,community_size_class,mean_num_clusters,mean_cluster_size,mean_min_cluster_size,mean_max_cluster_size
0,small,sparse,0.05,small,9.844,21.748047,7.980,37.300
1,small,sparse,0.05,large,4.948,44.037852,22.656,66.736
2,small,sparse,0.20,small,9.936,21.642124,6.708,37.688
3,small,sparse,0.20,large,5.220,41.859152,18.356,67.364
4,small,dense,0.05,small,10.148,21.114860,6.808,37.740
5,small,dense,0.05,large,5.456,39.980383,17.032,64.680
6,small,dense,0.20,small,10.428,20.724858,5.996,37.852
7,small,dense,0.20,large,5.552,39.291457,15.000,65.580
8,large,sparse,0.05,tiny,71.568,12.785026,2.120,27.076
9,large,sparse,0.05,small,8.780,118.053352,59.084,187.732


## Noise characteristics

In [8]:
def edge_fractions(edges: list[list[int]], ground_truth: list[list[int]]) -> tuple[float, float]:
    cluster_by_node = {
        node: cluster_id
        for cluster_id, cluster in enumerate(ground_truth)
        for node in cluster
    }

    internal_edges = sum(
        cluster_by_node[u] == cluster_by_node[v]
        for u, v in edges
    )

    external_edges = len(edges) - internal_edges

    possible_internal_edges = sum(
        len(cluster) * (len(cluster) - 1) // 2
        for cluster in ground_truth
    )

    external_fraction = external_edges / len(edges)

    missing_internal_fraction = (possible_internal_edges - internal_edges) / possible_internal_edges

    return external_fraction, missing_internal_fraction

In [9]:
instance_files = {path.stem: path for path in DATA_ROOT.rglob("*.json")}

instance_metadata = raw[GROUP_COLUMNS + ["instance", "p_in", "p_out"]].drop_duplicates(subset=["instance"]).copy()

structure_rows = []

for row in instance_metadata.itertuples(index=False):
    with instance_files[row.instance].open("r", encoding="utf-8") as file:
        data = json.load(file)

    external_fraction, missing_internal_fraction = edge_fractions(data["edges"], data["ground_truth"])

    structure_rows.append(
        {
            "instance": row.instance,
            "external_edge_fraction": external_fraction,
            "missing_internal_edge_fraction": missing_internal_fraction,
        }
    )

instance_metadata = instance_metadata.merge(pd.DataFrame(structure_rows), on="instance")

In [10]:
noise_summary = (
    instance_metadata
    .groupby(GROUP_COLUMNS, observed=True, as_index=False)
    .agg(
        mean_p_in=("p_in", "mean"),
        mean_p_out=("p_out", "mean"),
        mean_external_edge_fraction=("external_edge_fraction", "mean"),
        mean_missing_internal_edge_fraction=("missing_internal_edge_fraction", "mean"),
    )
)

noise_summary

,size_class,regime,noise,community_size_class,mean_p_in,mean_p_out,mean_external_edge_fraction,mean_missing_internal_edge_fraction
0,small,sparse,0.05,small,0.373388,0.002135,0.042472,0.625485
1,small,sparse,0.05,large,0.184291,0.002403,0.042457,0.815032
2,small,sparse,0.20,small,0.314432,0.008541,0.170626,0.685524
3,small,sparse,0.20,large,0.155193,0.009613,0.172712,0.844077
4,small,dense,0.05,small,0.746777,0.004271,0.042257,0.252562
5,small,dense,0.05,large,0.368583,0.004806,0.044353,0.631942
6,small,dense,0.20,small,0.628865,0.017082,0.172699,0.370996
7,small,dense,0.20,large,0.310385,0.019225,0.179433,0.689831
8,large,sparse,0.05,tiny,0.660870,0.000476,0.042212,0.338535
9,large,sparse,0.05,small,0.081876,0.000476,0.038306,0.918021


## Solution quality relative to the ground truth

For each algorithm and instance, the density ratio is defined as

$
\frac{d(P_{\mathrm{GT}})}{d(P_{\mathrm{algorithm}})}.
$

Interpretation:

- `1.0`: algorithm and ground truth have the same MDGP density,
- less than `1.0`: the algorithm finds a partition with higher MDGP density than the ground truth,
- greater than `1.0`: the algorithm finds a partition with lower MDGP density.

In [11]:
raw["density_ratio_to_ground_truth"] = raw["ground_truth_density"] / raw["density"]

quality_table = (
    raw
    .groupby(GROUP_COLUMNS + ["algorithm"], observed=True, as_index=False)
    .agg(mean_quality=("density_ratio_to_ground_truth", "mean"))
    .pivot(index=GROUP_COLUMNS, columns="algorithm", values="mean_quality")
    .reindex(columns=ALGORITHM_ORDER)
    .reset_index()
)

quality_table

algorithm,size_class,regime,noise,community_size_class,leiden,leiden_mdgp,kapoce,mdgp_plateau
0,small,sparse,0.05,small,1.020245,0.593937,0.553467,0.523221
1,small,sparse,0.05,large,1.001671,0.324482,0.291923,0.283674
2,small,sparse,0.20,small,1.057122,0.539093,0.475983,0.456736
3,small,sparse,0.20,large,1.017672,0.287039,0.251741,0.246905
4,small,dense,0.05,small,1.017746,1.005963,0.978956,0.884242
5,small,dense,0.05,large,1.007005,0.575148,0.521548,0.490432
6,small,dense,0.20,small,1.055919,0.988658,0.882436,0.773315
7,small,dense,0.20,large,1.030231,0.532521,0.446406,0.423690
8,large,sparse,0.05,tiny,1.273698,0.917095,0.915176,0.823549
9,large,sparse,0.05,small,1.000723,0.153211,0.134479,0.133613


## Ground-truth reconstruction using the clustering F-score

The second analysis measures how closely the partition produced by an algorithm reconstructs the planted ground-truth community structure.

Let $P_{\mathrm{GT}} = \{C_1^*, \dots, C_{k^*}^*\}$ denote the ground-truth partition and $P = \{C_1, \dots, C_k\}$ the partition produced by an algorithm.

For every pair consisting of a ground-truth cluster $C_i^*$ and an algorithmic cluster $C_j$, precision and recall are defined as

$P_{ij} = \frac{|C_i^* \cap C_j|}{|C_j|}$ and $R_{ij} = \frac{|C_i^* \cap C_j|}{|C_i^*|}$.

Their F-score is $F_{ij} = \frac{2P_{ij}R_{ij}}{P_{ij}+R_{ij}}$.

For every ground-truth cluster, only the best matching algorithmic cluster is considered. The overall clustering F-score is therefore

$
F(P, P_{\mathrm{GT}}) = \frac{1}{n} \sum_{i=1}^{k^*} |C_i^*| \max_{1 \leq j \leq k} F_{ij}.
$

The score lies between 0 and 1. A value of 1 indicates an exact reconstruction of the ground-truth partition, while lower values indicate increasing structural differences.

In [12]:
def cluster_f_score(ground_truth_cluster: set[int], cluster: set[int]) -> float:
    intersection = len(ground_truth_cluster & cluster)

    if intersection == 0:
        return 0.0

    return 2.0 * intersection / (len(ground_truth_cluster) + len(cluster))


def clustering_f_score(ground_truth_value: str, partition_value: str) -> float:
    ground_truth = [set(cluster) for cluster in json.loads(ground_truth_value)]
    partition = [set(cluster) for cluster in json.loads(partition_value)]

    n = sum(len(cluster) for cluster in ground_truth)

    return sum(
        len(ground_truth_cluster)
        * max(cluster_f_score(ground_truth_cluster, cluster) for cluster in partition)
        for ground_truth_cluster in ground_truth
    ) / n

In [13]:
raw["f_score"] = raw.apply(lambda row: clustering_f_score(row["ground_truth_partition"], row["partition"]), axis=1)

f_score_table = (
    raw
    .groupby(GROUP_COLUMNS + ["algorithm"], observed=True, as_index=False)
    .agg(mean_f_score=("f_score", "mean"))
    .pivot(index=GROUP_COLUMNS, columns="algorithm", values="mean_f_score")
    .reindex(columns=ALGORITHM_ORDER)
    .reset_index()
)

f_score_table

algorithm,size_class,regime,noise,community_size_class,leiden,leiden_mdgp,kapoce,mdgp_plateau
0,small,sparse,0.05,small,0.984203,0.291107,0.437192,0.333056
1,small,sparse,0.05,large,0.994615,0.153488,0.204149,0.172593
2,small,sparse,0.20,small,0.949439,0.260339,0.383836,0.309757
3,small,sparse,0.20,large,0.957628,0.143843,0.190730,0.167793
4,small,dense,0.05,small,0.986080,0.437919,0.959846,0.462973
5,small,dense,0.05,large,0.993776,0.189491,0.341219,0.220513
6,small,dense,0.20,small,0.956866,0.321757,0.816398,0.418860
7,small,dense,0.20,large,0.973837,0.158952,0.289526,0.207589
8,large,sparse,0.05,tiny,0.840299,0.496901,0.858481,0.557075
9,large,sparse,0.05,small,0.998318,0.065576,0.080312,0.072415


## LaTeX helper functions

In [ ]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor


def latex_algorithm(algorithm: str) -> str:
    return r"\texttt{" + algorithm.replace("_", r"\_") + "}"


def format_number(value: float, decimals: int) -> str:
    return f"{truncate_number(value, decimals):.{decimals}f}"


def format_percent(value: float, decimals: int) -> str:
    return f"{truncate_number(100 * value, decimals):.{decimals}f}" + r"\,\%"



## Build LaTeX tables

In [ ]:
def make_community_size_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    lines = [
        r"\begin{table}[!htbp]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\small",
        r"\begin{tabular}{p{2.0cm}rlrrrr}",
        r"\toprule",
        r"Dataset & $\mu$ & \shortstack{Community\\size} & \shortstack{Number of\\communities} & \shortstack{Mean\\size} & \shortstack{Mean\\minimum} & \shortstack{Mean\\maximum} \\",
        r"\midrule",
    ]

    for dataset_index, (size_class, regime) in enumerate(DATASET_ORDER):
        dataset_df = df[(df["size_class"] == size_class) & (df["regime"] == regime)]

        community_order = COMMUNITY_ORDER_BY_SIZE[size_class]
        dataset_row_count = len(NOISE_ORDER) * len(community_order)

        current_row = 0

        for noise_index, noise in enumerate(NOISE_ORDER):
            noise_df = dataset_df[np.isclose(dataset_df["noise"].astype(float), noise)]

            for community_index, community_size in enumerate(community_order):
                row = noise_df[noise_df["community_size_class"] == community_size].iloc[0]

                dataset_cell = (
                    rf"\multirow{{{dataset_row_count}}}{{*}}{{{size_class} {regime}}}"
                    if current_row == 0
                    else ""
                )

                noise_cell = (
                    rf"\multirow{{{len(community_order)}}}{{*}}{{{noise:.2f}}}"
                    if community_index == 0
                    else ""
                )

                lines.append(
                    f"{dataset_cell} "
                    f"& {noise_cell} "
                    f"& {community_size} "
                    f"& {row['mean_num_clusters']:.1f} "
                    f"& {row['mean_cluster_size']:.1f} "
                    f"& {row['mean_min_cluster_size']:.1f} "
                    f"& {row['mean_max_cluster_size']:.1f} "
                    r"\\"
                )

                current_row += 1

            if noise_index < len(NOISE_ORDER) - 1:
                lines.append(r"\cmidrule(l){2-7}")

        if dataset_index < len(DATASET_ORDER) - 1:
            lines.append(r"\midrule")

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ])

    return "\n".join(lines)

In [ ]:
community_size_latex = make_community_size_latex_table(
    community_size_summary,
    caption=(
        "Mean number and size of the ground-truth communities. Minimum and maximum denote the mean size of the smallest and largest community of an instance, respectively."
    ),
    label="tab:ground_truth_community_sizes",
)

print(community_size_latex)

In [ ]:
def make_degree_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    lines = [
        r"\begin{table}[H]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{lrr}",
        r"\toprule",
        r"Dataset & \shortstack{Mean target\\degree} & \shortstack{Mean realized\\vertex degree} \\",
        r"\midrule",
    ]

    for size_class, regime in DATASET_ORDER:
        row = df[(df["size_class"] == size_class) & (df["regime"] == regime)].iloc[0]

        lines.append(
            f"{size_class} {regime} "
            f"& {row['mean_target_degree']:.1f} "
            f"& {row['mean_realized_degree']:.1f} "
            r"\\"
        )

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ])

    return "\n".join(lines)

In [ ]:
degree_latex = make_degree_latex_table(
    degree_summary,
    caption=(
        "Mean target degree and mean realized average vertex degree of the ground-truth instances."
    ),
    label="tab:ground_truth_density",
)

print(degree_latex)

In [ ]:
def make_noise_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    lines = [
        r"\begin{table}[!htbp]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\small",
        r"\begin{tabular}{p{2.0cm}rlrrrr}",
        r"\toprule",
        r"Dataset & $\mu$ & \shortstack{Community\\size} & $p_{\mathrm{in}}$ & $p_{\mathrm{out}}$ & \shortstack{Fraction of\\external edges} & \shortstack{Fraction of missing\\internal edges} \\",
        r"\midrule",
    ]

    for dataset_index, (size_class, regime) in enumerate(DATASET_ORDER):
        dataset_df = df[(df["size_class"] == size_class) & (df["regime"] == regime)]

        community_order = COMMUNITY_ORDER_BY_SIZE[size_class]
        dataset_row_count = len(NOISE_ORDER) * len(community_order)

        current_row = 0

        for noise_index, noise in enumerate(NOISE_ORDER):
            noise_df = dataset_df[np.isclose(dataset_df["noise"].astype(float), noise)]

            for community_index, community_size in enumerate(community_order):
                row = noise_df[noise_df["community_size_class"] == community_size].iloc[0]

                dataset_cell = (
                    rf"\multirow{{{dataset_row_count}}}{{*}}{{{size_class} {regime}}}"
                    if current_row == 0
                    else ""
                )

                noise_cell = (
                    rf"\multirow{{{len(community_order)}}}{{*}}{{{noise:.2f}}}"
                    if community_index == 0
                    else ""
                )

                lines.append(
                    f"{dataset_cell} "
                    f"& {noise_cell} "
                    f"& {community_size} "
                    f"& {format_percent(row['mean_p_in'], 2)} "
                    f"& {format_percent(row['mean_p_out'], 2)} "
                    f"& {format_percent(row['mean_external_edge_fraction'], 2)} "
                    f"& {format_percent(row['mean_missing_internal_edge_fraction'], 2)} "
                    r"\\"
                )

                current_row += 1

            if noise_index < len(NOISE_ORDER) - 1:
                lines.append(r"\cmidrule(l){2-7}")

        if dataset_index < len(DATASET_ORDER) - 1:
            lines.append(r"\midrule")

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ])

    return "\n".join(lines)

In [ ]:
noise_latex = make_noise_latex_table(
    noise_summary,
    caption=(
        "Mean edge probabilities and mean realized fractions of external and missing internal edges of the ground-truth communities."
    ),
    label="tab:ground_truth_noise",
)

print(noise_latex)

In [ ]:
def make_algorithm_latex_table(df: pd.DataFrame, caption: str, label: str, maximize: bool) -> str:
    lines = [
        r"\begin{table}[!htbp]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\small",
        r"\begin{tabular}{p{2.0cm}rlrrrr}",
        r"\toprule",
        r"Dataset & $\mu$ & \shortstack{Community\\size} & \texttt{Leiden} & \shortstack{\texttt{Leiden-}\\\texttt{MDGP}} & \texttt{KaPoCE} & \shortstack{\textls[-50]{\textsc{MDGP-}}\\\textls[-50]{\textsc{Plateau}}} \\",
        r"\midrule",
    ]

    for dataset_index, (size_class, regime) in enumerate(DATASET_ORDER):
        dataset_df = df[(df["size_class"] == size_class) & (df["regime"] == regime)]

        community_order = COMMUNITY_ORDER_BY_SIZE[size_class]
        dataset_row_count = len(NOISE_ORDER) * len(community_order)

        current_row = 0

        for noise_index, noise in enumerate(NOISE_ORDER):
            noise_df = dataset_df[np.isclose(dataset_df["noise"].astype(float), noise)]

            for community_index, community_size in enumerate(community_order):
                row = noise_df[noise_df["community_size_class"] == community_size].iloc[0]

                values = {algorithm: row[algorithm] for algorithm in ALGORITHM_ORDER}

                best_value = max(values.values()) if maximize else min(values.values())

                formatted = {algorithm: format_number(value, 4) for algorithm, value in values.items()}

                for algorithm, value in values.items():
                    if np.isclose(value, best_value):formatted[algorithm] = (rf"\textbf{{{formatted[algorithm]}}}")

                dataset_cell = (
                    rf"\multirow{{{dataset_row_count}}}{{*}}{{{size_class} {regime}}}"
                    if current_row == 0
                    else ""
                )

                noise_cell = (
                    rf"\multirow{{{len(community_order)}}}{{*}}{{{noise:.2f}}}"
                    if community_index == 0
                    else ""
                )

                lines.append(
                    f"{dataset_cell} "
                    f"& {noise_cell} "
                    f"& {community_size} "
                    f"& {formatted['leiden']} "
                    f"& {formatted['leiden_mdgp']} "
                    f"& {formatted['kapoce']} "
                    f"& {formatted['mdgp_plateau']} "
                    r"\\"
                )

                current_row += 1

            if noise_index < len(NOISE_ORDER) - 1:
                lines.append(r"\cmidrule(l){2-7}")

        if dataset_index < len(DATASET_ORDER) - 1:
            lines.append(r"\midrule")

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ])

    return "\n".join(lines)

In [ ]:
quality_latex = make_algorithm_latex_table(
    quality_table,
    caption=(
        "Mean relative solution quality regarding the ground-truth partition."
    ),
    label="tab:ground_truth_quality",
    maximize=False,
)

print(quality_latex)

In [ ]:
f_score_latex = make_algorithm_latex_table(
    f_score_table,
    caption=(
        "Mean clustering F-score of the considered algorithms regarding the ground-truth partition. Higher values indicate stronger structural agreement with the ground truth."
    ),
    label="tab:ground_truth_f_score",
    maximize=True,
)

print(f_score_latex)